# Exploratory Data Analysis for Fungi Dataset

In [ ]:
!which python3

In [ ]:
import pandas as pd
import os

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
data_path = '/Users/timothysalmon/Code/projects/mushroom_id/data/0055956-241126133413365'
files = ['citations.txt', 'meta.xml', 'multimedia.txt', 'rights.txt', 'datasetmetadata.xml', 'occurrence.txt', 'verbatim.txt']


In [ ]:
occurrence = pd.read_csv(os.path.join(data_path, 'occurrence.txt'), sep="\t")

In [ ]:
multimedia = pd.read_csv(os.path.join(data_path, 'multimedia.txt'), sep="\t")

In [ ]:
# print('\n'.join(occurrence.columns.to_list()))

In [ ]:
occurrence.head()

In [ ]:
multimedia.head()

In [ ]:
occurrence_ids = set(occurrence['gbifID'].tolist())
multimedia_ids = set(multimedia['gbifID'].tolist())

In [ ]:
len(occurrence_ids), len(multimedia_ids), len(occurrence_ids.intersection(multimedia_ids)), len(occurrence_ids.intersection(multimedia_ids)) / len(occurrence_ids)

In [ ]:
# identifier seems to contain a link to the actual image
multimedia['identifier'][0:5].to_list()

In [ ]:
verbatim = pd.read_csv(os.path.join(data_path, 'verbatim.txt'), sep="\t")

In [ ]:
# genus
occurrence['genus'].value_counts().to_dict()

In [ ]:
occurrence['genus'].value_counts().sum()

In [ ]:
chantarelle_genera = ['Cantharellus', 'Craterellus', 'Gomphus', 'Polyozellus']

In [ ]:
for genus in chantarelle_genera:
    print(occurrence['genus'].value_counts()[genus])

In [ ]:
(occurrence['genus'].value_counts() >= 100).value_counts()

## Observation
### 657 genera have at least 100 observations associated with them. Let's bin the remaining genera into an "OTHER" category and try to predict the 657 genera

In [ ]:
low_count_genera = [k for k,v in (occurrence['genus'].value_counts() < 250).to_dict().items() if v]

In [ ]:
occurrence['genus'].value_counts().loc[low_count_genera].sum() / occurrence['genus'].value_counts().sum()

In [ ]:
genus_by_gbfID = occurrence[['gbifID', 'genus']].set_index('gbifID')
image_by_gbfID = multimedia[['gbifID', 'identifier']].set_index('gbifID')

In [ ]:
merged = genus_by_gbfID.join(image_by_gbfID, how='inner').reset_index()

In [ ]:
low_count_threshold = 250

# 1. Compute counts of each genus
counts = merged['genus'].value_counts()

# 2. Identify genera occurring fewer than `low_count_threshold` times
low_count_genera = counts[counts < low_count_threshold].index

# 3. Create a relabeled column
merged['genus'] = merged['genus'].apply(
    lambda g: 'Unknown_Genus' if g in low_count_genera else g
)

# Note for later: Some of these images contain selfies of the mushrooms foragers. Will need to filter those out, but face detection seems annoying so we'll skip for now

In [ ]:
merged.head()

In [ ]:
merged['genus'].value_counts()

In [ ]:
merged['identifier'].str.contains('https://inaturalist-open-data.s3.').value_counts()

In [1]:
# estimate of total image volume. Naively estimate that each file is 64KB (Which it should be after preprocess). 
f'{3460020 * 64 / 1024 / 1024 / 1024} TB'
## Full Dataset should be 200 GB when fully downloaded then

'0.20623326301574707 TB'

In [ ]:
df = merged
zarr_path = 'data/fungi.zarr'

In [ ]:
import os
import zarr
import asyncio
import aiohttp
from aiohttp import ClientTimeout, TCPConnector
from io import BytesIO
from PIL import Image
import numpy as np
import pandas as pd

def build_zarr_store(df: pd.DataFrame, zarr_path: str, 
                     batch_size: int = 1000,
                     max_concurrency: int = 200):
    """
    Download all images in df['identifier'], resize to 256×256 RGB,
    and store in a Zarr array, along with df['gbifID'] and df['genus'].
    """

    # ————— Prepare Zarr —————
    os.makedirs(os.path.dirname(zarr_path), exist_ok=True)
    n = len(df)
    root = zarr.open_group(zarr_path, mode='w')

    images = root.create_array(
        name='images',
        shape=(n, 3, 256, 256),
        chunks=(batch_size, 3, 256, 256),
        dtype='u1',
        overwrite=True,
    )
    gbif = root.create_array('gbifID', shape=(n,), dtype='i8', overwrite=True)
    gbif[:] = df['gbifID'].values.astype('i8')

    genus_str = df['genus'].fillna('').astype(str)
    maxlen = int(genus_str.str.len().max())
    genus = root.create_array('genus', shape=(n,), dtype=f'|S{maxlen}', overwrite=True)
    genus_bytes = (
        genus_str.str.encode('utf-8')
                 .apply(lambda b: b.ljust(maxlen, b'\0')[:maxlen])
                 .values
    )
    genus[:] = genus_bytes

    # ————— Async downloader setup —————
    timeout = ClientTimeout(total=10)
    connector = TCPConnector(limit=max_concurrency, force_close=True)
    session_args = dict(timeout=timeout, connector=connector, http2=True)
    failed = []

    async def fetch(i, url):
        try:
            async with aiohttp.ClientSession(**session_args) as sess:
                async with sess.get(url) as resp:
                    if resp.status == 404:
                        raise aiohttp.ClientResponseError(
                            status=404,
                            request_info=resp.request_info,
                            history=resp.history,
                            message="Not Found"
                        )
                    resp.raise_for_status()
                    data = await resp.read()
            img = Image.open(BytesIO(data)).convert('RGB')
            img = img.resize((256, 256), Image.BILINEAR)
            return i, np.asarray(img, dtype=np.uint8).transpose(2, 0, 1)
        except Exception as e:
            failed.append((i, url, repr(e)))
            return i, np.zeros((3, 256, 256), dtype=np.uint8)

    async def download_batch(indices, urls):
        tasks = [asyncio.create_task(fetch(i, url))
                 for i, url in zip(indices, urls)]
        results = []
        for coro in asyncio.as_completed(tasks):
            results.append(await coro)
        return results

    # ————— Download in batches & write —————
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        idxs = list(range(start, end))
        urls = df['identifier'].iloc[start:end].tolist()

        # create a fresh loop so we can run in Jupyter (or any active-loop env)
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        batch_results = loop.run_until_complete(download_batch(idxs, urls))
        loop.close()

        # pack and write once
        batch_arr = np.stack([arr for _, arr in sorted(batch_results)], axis=0)
        images[start:end] = batch_arr

    # ————— Summary —————
    if failed:
        print(f"\nWarning: {len(failed)} images failed. Sample:")
        for i, url, err in failed[:5]:
            print(f"  [{i}] {url} → {err}")

    print("Zarr store built at", zarr_path)

In [ ]:
print(os.path.exists(zarr_path))

In [ ]:
if not os.path.exists(zarr_path):
    build_zarr_store(df.sample(1000, axis=0), zarr_path) # only grab first 1000 images
root = zarr.open(zarr_path, mode='r')

In [ ]:
print('test')

In [ ]:
len(df) // 100

In [ ]:
import zarr
import matplotlib.pyplot as plt
import numpy as np

# 1. Open the Zarr store:
store_path = 'data/fungi.zarr'  # adjust to your path
z = zarr.open(store_path, mode='r')

# 2. Read the images and GBIF arrays:
images = z['images']       # shape: (N, 3, 256, 256)
gbif_ids = z['gbifID'][:]  # shape: (N,)

# 3. Sample a few indices for sanity-check:
np.random.seed(0)
sample_idxs = np.random.choice(images.shape[0], size=9, replace=False)

# 4. Plot a 3×3 grid of images:
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, idx in zip(axes.flatten(), sample_idxs):
    img = images[idx]                   # (3,256,256)
    img = np.transpose(img, (1, 2, 0))  # → (256,256,3)
    ax.imshow(img)
    ax.set_title(f"Idx {idx}\nGBIF {gbif_ids[idx]}")
    ax.axis('off')

plt.tight_layout()
plt.show()